# Setup

In [1]:
import torch
import numpy as np
# from imgaug import augmenters as iaa   # not used, and breaks on newer numpy
import torchvision
import PIL.Image as Image
import cv2
import math

from blindMatch import *

Congratulations on [BlindAuth] file import!


# Data Generator


In [2]:
class FingerprintDataset(torch.utils.data.Dataset):
    def __init__(self, meta, degree, size=224):
        with open(meta, 'r') as fin:
            self.x = [x for x in fin]

        self.transform = torchvision.transforms.Compose([
            torchvision.transforms.Resize((size, size)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.RandomRotation(degree),
            torchvision.transforms.Normalize((0.5, 0.5, 0.5), (1.0, 1.0, 1.0))
        ])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        entry = self.x[idx].split('|')
        # Write down your data path
        root = '../data/PolyU_Dataset/'

        file = Image.fromarray(cv2.imread(root + entry[0]))
        file = self.transform(file)

        n_id = entry[1]

        return file, np.array(n_id, dtype=np.int64)


In [3]:
batch = 64

trainset = FingerprintDataset('polyu_meta.txt', 0)        # was '../polyu_meta.txt'
testset  = FingerprintDataset('polyu_meta_eval.txt', 0)   # was '../polyu_meta_eval.txt'

dataloader = torch.utils.data.DataLoader(trainset, batch_size=batch, shuffle=True, drop_last=True, num_workers=0)
dataloader_val = torch.utils.data.DataLoader(testset, batch_size=batch, shuffle=False, drop_last=False, num_workers=0)

print(len(trainset), "train images |", len(testset), "eval images")

2016 train images | 960 eval images


# Trainining the model

In [4]:
import time   # ADDED

c = 8
num_ids = 400
device = 'cuda'

# network selecting - ST/Resnet18
net = FingerSTNNet(16*c) # net = FingerNet(16*c)
net = net.to(device)
print("Feature size", 16*c, "STNet")

fc = FingerCentroids(num_ids, 16*c)
fc = fc.to(device)

loss_arcface = ArcFace(m=0.2)

optim = torch.optim.Adam(net.parameters(), lr=0.001)
degree = 20
best_acc = 0
best_loss = 100

for epoch in range(2):        # CHANGED: was 100 -> test with 2 first
    t0 = time.time()          # ADDED
    net = net.train()
    loss_accum = []

    for idx, (img, lbl) in enumerate(dataloader):
        img = img.to(device)
        lbl = lbl.to(device)

        feat = net(img)
        logit = fc(feat)
        logit = loss_arcface(logit, lbl)

        loss = torch.nn.functional.cross_entropy(logit, lbl)
        loss_accum.append(loss.item())

        loss.backward()
        optim.step()
        optim.zero_grad()

        if idx % 2 == 0:
            print('.', end='')

    loss_accum = torch.tensor(loss_accum)
    print(f'epoch: {epoch} | loss: {loss_accum.mean().item():.04f} | {time.time()-t0:.1f}s')  # CHANGED: added time

    net = net.eval()

    eval_results = []
    lbls = []

    with torch.no_grad():
        for idx, (img, lbl) in enumerate(dataloader_val):
            img = img.to(device)
            feat = net(img)

            eval_results.append(feat.to('cpu'))
            lbls.append(lbl)

        eval_results = torch.cat(eval_results)
        mat_similarity = eval_results.matmul(eval_results.T)

        lbls = torch.cat(lbls)
        lbls = lbls.view(-1, lbls.size(0)) == lbls.view(lbls.size(0), -1)

        accuracy = []

        total_comp = torch.ones_like(mat_similarity).triu(1)
        total_comp = total_comp.sum().item()

        thresh_best_acc = 0
        for threshold in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:   # CHANGED: 0,8 -> 0.8
            threshed = mat_similarity > threshold

            #remove diagonal
            correct = (threshed == lbls).triu(1).sum()

            accuracy.append(correct / total_comp)
            if accuracy[-1] > thresh_best_acc:
                thresh_best_acc = accuracy[-1]

        print(f'Accuracy: {" | ".join(f"{acc:.03f}" for acc in accuracy)}')

        if best_acc < thresh_best_acc:
            best_acc = thresh_best_acc
            best_loss = loss_accum.mean()
            best_epoch = epoch
            best_sim = mat_similarity
            torch.save(net.state_dict(), 'best_net.pth')   # ADDED: save the best model

    print('=' * 20)

Feature size 128 STNet


c:\Users\Admin\blind-match\training\blindMatch.py:56: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  grid = torch.nn.functional.affine_grid(theta, x.size())
c:\Users\Admin\blind-match\training\blindMatch.py:57: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  x = torch.nn.functional.grid_sample(x, grid)


................epoch: 0 | loss: 23.6445 | 17.5s
Accuracy: 0.005 | 0.005 | 0.005 | 0.005 | 0.005 | 0.005 | 0.066 | 0.226 | 0.380 | 0.575
................epoch: 1 | loss: 20.8157 | 7.4s
Accuracy: 0.005 | 0.005 | 0.005 | 0.005 | 0.033 | 0.231 | 0.478 | 0.586 | 0.682 | 0.794


In [5]:
print("MOST", best_loss, best_acc, best_epoch)

accuracy = []
for threshold in [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:   # CHANGED: 0,8 -> 0.8
    threshed = best_sim > threshold

    #remove diagonal
    correct = (threshed == lbls).triu(1).sum()

    accuracy.append(correct / total_comp)
    if accuracy[-1] > thresh_best_acc:
        thresh_best_acc = accuracy[-1]

print("MOST", best_loss, best_acc, thresh_best_acc)
print(f'Accuracy: {" | ".join(f"{acc:.05f}" for acc in accuracy) } \n Loss: {"".join(f"{loss_accum.mean():.05f}")}')
print("".join(f"{accuracy[-1]:.05f}"), "".join(f"{loss_accum[-1]:.05f}"))
print( 16*c, net)

MOST tensor(20.8157) tensor(0.7945) 1
MOST tensor(20.8157) tensor(0.7945) tensor(0.7945)
Accuracy: 0.00521 | 0.00521 | 0.00521 | 0.00521 | 0.03257 | 0.23129 | 0.47763 | 0.58622 | 0.68197 | 0.79446 
 Loss: 20.81571
0.79446 19.92252
128 FingerSTNNet(
  (basenet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e